# Spatial weights: build the neighbourhood before the statistic

This lab shows how the same places can produce different neighbour graphs. We will compare Rook and Queen contiguity, row-standardised weights, distance bands, k-nearest neighbours and isolates.

**You need:** Python, NumPy, Matplotlib, Shapely and libpysal.

The four polygons form a simple 2-by-2 grid. This makes corner-only and edge-sharing relationships easy to see.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from shapely import box
from shapely.geometry import Point
from libpysal import graph


## 1. Create four neighbouring polygons

Cells A and D touch at one corner. Cells B and C also touch at one corner. Rook contiguity excludes those diagonal pairs; Queen contiguity includes them.

In [ ]:
geometries = [
    box(0, 1, 1, 2),  # A, upper left
    box(1, 1, 2, 2),  # B, upper right
    box(0, 0, 1, 1),  # C, lower left
    box(1, 0, 2, 1),  # D, lower right
]

rook = graph.Graph.build_contiguity(geometries, rook=True)
queen = graph.Graph.build_contiguity(geometries, rook=False)

def graph_matrix(weights, n):
    matrix = np.zeros((n, n), dtype=float)
    for (focal, neighbor), value in weights.adjacency.items():
        matrix[int(focal), int(neighbor)] = float(value)
    return matrix

rook_matrix = graph_matrix(rook, len(geometries))
queen_matrix = graph_matrix(queen, len(geometries))

print('Rook adjacency matrix:')
print(rook_matrix)
print('Queen adjacency matrix:')
print(queen_matrix)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
labels = ['A', 'B', 'C', 'D']
for i, geometry in enumerate(geometries):
    minx, miny, maxx, maxy = geometry.bounds
    ax.add_patch(Rectangle((minx, miny), maxx - minx, maxy - miny, fill=False, linewidth=2))
    ax.text((minx + maxx) / 2, (miny + maxy) / 2, labels[i], ha='center', va='center', fontsize=16)
ax.set_xlim(-0.1, 2.1)
ax.set_ylim(-0.1, 2.1)
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('A 2-by-2 polygon grid')
plt.show()


## 2. Compare the adjacency matrices

The only difference between these two matrices is the pair of diagonal corner-touching relationships. The spatial units have not changed; the neighbour definition has.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, matrix, title in zip(axes, [rook_matrix, queen_matrix], ['Rook', 'Queen']):
    image = ax.imshow(matrix, vmin=0, vmax=1)
    ax.set_xticks(range(4), labels)
    ax.set_yticks(range(4), labels)
    ax.set_xlabel('Neighbour')
    ax.set_ylabel('Focal unit')
    ax.set_title(title)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{matrix[i, j]:.0f}', ha='center', va='center')
plt.tight_layout()
plt.show()


## 3. Row-standardise the weights

Row standardisation makes each nonzero row sum to one. An isolate stays a zero row. The transformation changes the weights, so record it whenever you report a spatial statistic.

In [ ]:
queen_row = queen.transform('r')
queen_row_matrix = graph_matrix(queen_row, len(geometries))
print(queen_row_matrix)
print('Row sums:', queen_row_matrix.sum(axis=1))


## 4. Compare distance bands and k-nearest neighbours

The centroids of the four cells form a small point pattern. A distance band connects points within a threshold. A k-nearest-neighbour graph connects each point to a fixed number of closest points; it may be asymmetric.

In [ ]:
centroids = np.array([geometry.centroid for geometry in geometries], dtype=object)
distance_band = graph.Graph.build_distance_band(centroids, threshold=1.01)
knn = graph.Graph.build_knn(centroids, k=1)

distance_matrix = graph_matrix(distance_band, len(geometries))
knn_matrix = graph_matrix(knn, len(geometries))

print('Distance-band graph (threshold 1.01):')
print(distance_matrix)
print('kNN graph (k=1):')
print(knn_matrix)
print('Is kNN symmetric?', np.array_equal(knn_matrix, knn_matrix.T))


centroids = np.array([geometry.centroid for geometry in geometries], dtype=object)
distance_band = graph.Graph.build_distance_band(centroids, threshold=1.01)

knn_points = np.array([Point(0, 0), Point(1, 0), Point(3, 0), Point(10, 0)], dtype=object)
knn = graph.Graph.build_knn(knn_points, k=1)

distance_matrix = graph_matrix(distance_band, len(geometries))
knn_matrix = graph_matrix(knn, len(knn_points))

print('Distance-band graph (threshold 1.01):')
print(distance_matrix)
print('kNN graph (k=1, points at 0, 1, 3, 10):')
print(knn_matrix)
print('Is kNN symmetric?', np.array_equal(knn_matrix, knn_matrix.T))


In [ ]:
too_small = graph.Graph.build_distance_band(centroids, threshold=0.25)
print('Isolated units:', list(too_small.isolates))
print('Graph summary:')
print(too_small.summary())


## 6. Row standardisation can create asymmetry

The Queen graph above has the same number of neighbours for every unit, so row standardisation keeps it symmetric. A three-cell line has different degrees at its ends and centre. After row standardisation, the outgoing weights differ.

In [ ]:
line_geometries = [box(0, 0, 1, 1), box(1, 0, 2, 1), box(2, 0, 3, 1)]
line_graph = graph.Graph.build_contiguity(line_geometries, rook=True)
line_row = line_graph.transform('r')
line_binary_matrix = graph_matrix(line_graph, 3)
line_row_matrix = graph_matrix(line_row, 3)

print('Binary line graph:')
print(line_binary_matrix)
print('Row-standardised line graph:')
print(line_row_matrix)
print('Is row-standardised graph symmetric?', np.array_equal(line_row_matrix, line_row_matrix.T))


## What to take away

- A weights matrix makes the meaning of neighbour explicit.
- Queen includes corner-touching polygons; Rook does not.
- Distance bands depend on a threshold and meaningful coordinate units.
- k-nearest-neighbour graphs can be asymmetric.
- Row standardisation changes weights and can introduce asymmetry.
- Isolates must be identified and reported.

Continue with the [spatial weights lesson](../methods/spatial-weights-matrix.md), the [Global Moran's I lesson](../algorithms/statistics/global-morans-i.md) and the [spatial autocorrelation learning path](../learning-paths/global-spatial-autocorrelation.md).

**Reference:** [libpysal Graph API](https://pysal.org/libpysal/stable/generated/libpysal.graph.Graph.html).